# ASA Decoder Evaluation

Measures **Decode & Infer** on its own — one sentence in, one affect vector out — with no agent
around it. No affect model, no clock, no evidence loop, no observer registry and no `run_agent`.

**Why the decoder can be measured in isolation at all.** `AffectDecoder` is stateless by contract:
it sees the utterance and nothing else, so the same sentence decodes to the same vector whenever it
arrives, whatever was said before. Scoring is therefore a loop over rows rather than a session, and
what comes out is a property of the **instrument** rather than of one run through it.

**What this buys over `evaluation_runs.ipynb`.** That notebook replays a benchmark through the live
agent and scores what came out the far end, which is the right way to ask whether the *pipeline*
works. Three things fall away when only the decoder is under test:

- no `gap` between utterances, so a full corpus takes seconds rather than hours;
- no fold, no decay and no belief, so nothing stands between the decoder's answer and the score;
- no `intended` round trip. The replay path has to turn ground truth into an `AffectVector` so it
  can travel on an `Utterance`, which forces every unannotated axis to a real magnitude, and then
  needs `unmeasured` to put the NaNs back afterwards. Here the labels never leave the frame.

**What it cannot tell you**: anything about accumulation — whether a belief settles sensibly, how
decay behaves between turns, what the agent ends up expressing. Those need the whole cycle, and
`evaluation_runs.ipynb` is where they are asked.

## Shape of the notebook

1. **Benchmarks** — the labelled corpora, keyed by the name each one carries.
2. **Conditions** — the instruments under test: a representation, a table and a decoder.
3. **Harness** — one condition over one benchmark, to a frame of predictions beside truth.
4. **Scoring** — multi-label decisions at a threshold: micro, macro and per axis.
5. **One condition over one benchmark** — the single run, and its threshold sweep.
6. **Comparison** — the grid over conditions x benchmarks x thresholds.
7. **Diagnostic** — what `plutchik8/1` sees that `ekman6/1` has no axis for.
8. **Error analysis** — what fired wrongly, what stayed silent and why.
9. **Persisting** — a result written to `data_results/` with its provenance attached.


In [1]:
# ASA Imports
# Notebook Specifics / Temporary Functions
#
import logging

from asa._tools.custom_logging import setup_logging
from asa._tools.paths import repo_root

log = logging.getLogger(f"{__name__}.app")
setup_logging(level="INFO")

DATA_IN = repo_root() / "data_in"
DATA_RESULTS = repo_root() / "data_results"

# INFO rather than DEBUG: there is no run to watch here. The per-record log in
# `evaluation_runs.ipynb` exists because a replay has an order and a clock worth seeing; a
# decode sweep has neither, and 8,522 DEBUG lines per condition would bury the results.
log.info("Notebook Decoder Evaluation")


INFO: __main__.app.<module>.line_18 - Notebook Decoder Evaluation


---
## 1. Benchmarks

The prepared frames from `data_load_benchmarks.ipynb`, in the standard shape: `source`, `split`,
`id`, `text` and one float column per axis of the representation. NaN in an axis column means
**nobody looked**, which is a different claim from `0.0` — the annotators looked and the emotion was
not there — and keeping the two apart is what stops a scorer counting every prediction on an
unlabelled class as a false positive.


In [2]:
# The labelled corpora, keyed by the name each frame carries in its own attrs
#
# Keyed on `attrs["corpus"]` rather than on the filename, so a renamed or re-saved file cannot
# quietly relabel a result. The frame names itself; the notebook only reads that name.

import pandas as pd

BENCHMARK_FILES = (DATA_IN / "bench_simple_ekman6.parquet",
                   DATA_IN / "bench_brighter_eng.parquet")

benchmarks = {}
for path in BENCHMARK_FILES:
    frame = pd.read_parquet(path)
    benchmarks[frame.attrs["corpus"]] = frame

display(pd.DataFrame([{"corpus": name,
                       "rows": len(frame),
                       "representation": frame.attrs["representation"],
                       "unannotated": ", ".join(frame.attrs["unannotated_axes"]) or "-",
                       "axis_map": frame.attrs["axis_map"] or "-"}
                      for name, frame in benchmarks.items()]))

display(benchmarks["simple-ekman6"].head(5))
display(benchmarks["brighter-eng"].head(5))


,corpus,rows,representation,unannotated,axis_map
0,simple-ekman6,20,ekman6/1,-,-
1,brighter-eng,8522,ekman6/1,disgust,{'joy': 'happiness'}


,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,simple-ekman6,all,simple-ekman6_00000,I am so happy,0.0,0.0,0.0,1.0,0.0,0.0
1,simple-ekman6,all,simple-ekman6_00001,I'm absolutely delighted,0.0,0.0,0.0,1.0,0.0,0.0
2,simple-ekman6,all,simple-ekman6_00002,I was gutted,0.0,0.0,0.0,0.0,1.0,0.0
3,simple-ekman6,all,simple-ekman6_00003,I feel miserable today,0.0,0.0,0.0,0.0,1.0,0.0
4,simple-ekman6,all,simple-ekman6_00004,that is absolutely revolting,0.0,1.0,0.0,0.0,0.0,0.0


,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,brighter-eng,train,eng_train_track_a_00001,"colorado, middle of nowhere.",0.0,NaN,1.0,0.0,0.0,1.0
1,brighter-eng,train,eng_train_track_a_00002,this involved swimming a pretty large lake tha...,0.0,NaN,1.0,0.0,0.0,0.0
2,brighter-eng,train,eng_train_track_a_00003,it was one of my most shameful experiences.,0.0,NaN,1.0,0.0,1.0,0.0
3,brighter-eng,train,eng_train_track_a_00004,"after all, i had vegetables coming out my ears...",0.0,NaN,0.0,0.0,0.0,0.0
4,brighter-eng,train,eng_train_track_a_00005,then the screaming started.,0.0,NaN,1.0,0.0,1.0,1.0


---
## 2. Conditions — the instruments under test

A **condition** is one decoder plus the representation its output is written in. Everything the
comparison varies lives here, so adding an instrument is one entry in `CONDITIONS` and nothing
downstream changes — which is the point, since the LLM decoder is meant to arrive as a fourth entry
rather than as a second notebook.


In [3]:
# One decoder, named, with the representation its answers are written in
#

from dataclasses import dataclass

from asa.core.representations import EKMAN6, PLUTCHIK8, AffectRepresentation
from asa.perception.base import AffectDecoder
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, LEXICON_HANDWRITTEN, KeywordDecoder
from asa.perception.nrc_eil import load_tables

EIL_LEXICON = DATA_IN / "NRC-Emotion-Intensity-Lexicon-v1.loaded-2026-08-13.json"


@dataclass(frozen=True)
class Condition:
    """One instrument under test: a decoder, the representation it speaks and a label.

    **The representation travels with the decoder because the decoder will not say.**
    `AffectDecoder` is a single method returning an observation, and the vector inside names its
    representation as a bare string — deliberately, so a recorded row naming a representation this
    build no longer declares still loads. Scoring needs the descriptor itself: the axis order, the
    `rest` value that decides what counts as fired, and `metric`, which decides whether averaging
    across axes is even legitimate.

    **`name` is this notebook's label, and it is NOT the decoder's `source`.** The source string is
    provenance — `decoder:rule:handwritten`, composed by the decoder from its algorithm and its
    table — and it is recorded per row by the harness. Keeping them apart means a saved result can
    be traced back to the instrument that produced it after the notebook's labels have been
    rewritten, which they will be.
    """

    name: str
    decoder: AffectDecoder
    representation: AffectRepresentation
    note: str = ""


# The prepared lexicon artefact. A fresh clone will not have it — `data_in/` is untracked and the
# NRC lexicon may not be redistributed — so run `data_load_lexicons.ipynb` first.
lexicon = load_tables(EIL_LEXICON, [PLUTCHIK8, EKMAN6])

print(f"lexicon source : {lexicon.source}")
for rep_id, table in lexicon.tables.items():
    entries = sum(len(words) for words in table.values())
    distinct = len({word for words in table.values() for word in words})
    print(f"{rep_id:<12} {len(table)} axes, {entries:,} entries, {distinct:,} distinct words")

CONDITIONS = (
    Condition(name="rule/handwritten",
              decoder=KeywordDecoder(representation=EKMAN6,
                                     table=EKMAN6_KEYWORDS,
                                     lexicon=LEXICON_HANDWRITTEN),
              representation=EKMAN6,
              note="iteration 1's shipped decoder - a few dozen hand-set words"),
    Condition(name="rule/nrc-eil",
              decoder=KeywordDecoder(representation=EKMAN6,
                                     table=lexicon.tables[EKMAN6.id],
                                     lexicon=lexicon.source),
              representation=EKMAN6,
              note="same algorithm, NRC-EIL restricted to ekman6/1"),
    Condition(name="rule/nrc-eil-plutchik8",
              decoder=KeywordDecoder(representation=PLUTCHIK8,
                                     table=lexicon.tables[PLUTCHIK8.id],
                                     lexicon=lexicon.source),
              representation=PLUTCHIK8,
              note="DIAGNOSTIC - anticipation and trust have no ground truth in any benchmark"),
)

display(pd.DataFrame([{"condition": c.name, "representation": c.representation.id, "note": c.note}
                      for c in CONDITIONS]))


lexicon source : nrc-eil
plutchik8/1  8 axes, 9,824 entries, 5,891 distinct words
ekman6/1     6 axes, 7,472 entries, 4,689 distinct words


,condition,representation,note
0,rule/handwritten,ekman6/1,iteration 1's shipped decoder - a few dozen ha...
1,rule/nrc-eil,ekman6/1,"same algorithm, NRC-EIL restricted to ekman6/1"
2,rule/nrc-eil-plutchik8,plutchik8/1,DIAGNOSTIC - anticipation and trust have no gr...


---
## 3. The harness

One condition over one benchmark frame, to one row per utterance carrying both what the decoder said
and what the corpus says is true.


In [4]:
# Decode every row of a benchmark, and put the answers beside the labels
#

import time

from asa.core.affect import Utterance

SOURCE = "input:decoder_eval"


async def decode_all(frame: pd.DataFrame, condition: Condition) -> pd.DataFrame:
    """One condition over one benchmark frame -> one row per utterance, predictions beside truth.

    **A real `Utterance` per row, through the real port.** The decoder is handed exactly the type it
    is handed in a live session and answers through the same `decode`, so a result here transfers to
    the agent rather than describing a notebook-only path.

    **`intended` is deliberately left unset.** Ground truth stays in the frame as columns and is
    copied across at the end, so an unannotated axis is NaN the whole way through. The replay path
    cannot do that — an `AffectVector` has no NaN — which is why it needs `assume_absent` going in
    and `unmeasured` coming out, two chances to disagree about the same fact. Here there is one.

    **`true_` columns are built for every axis of the DECODER's representation**, filled with NaN
    where the benchmark has no such column. So a `plutchik8/1` decoder scored against an `ekman6/1`
    corpus gets `true_anticipation` and `true_trust` full of NaN, and the scorer drops them under the
    same all-NaN rule that drops BRIGHTER-eng's `disgust`. Nothing has to name a corpus or a
    representation for that to work.

    **Keyed on the corpus's own `id`, not the utterance's.** `Utterance.id` is minted per run and
    means nothing outside it; a row that decoded badly has to be lookupable in the corpus it came
    from.

    **A raising decoder is recorded, never dropped.** `run_perception` logs the exception and carries
    on, which costs one evidence row in a live session; here it would silently shrink the
    denominator, and a decoder that fails on a third of the corpus and does well on the rest would
    look excellent. A failure fills `error` and leaves every `pred_` at NaN — which scores as
    predicting nothing, the honest reading — and the count is carried in `attrs` so it cannot be
    missed.

    **Timed per row off the monotonic clock.** The decode sits inside the roughly one-second
    end-of-input-to-expression budget, so latency is a property of the instrument worth comparing —
    and it is the number that changes by orders of magnitude the day the decoder becomes an LLM call.

    **Sequential, one `await` at a time.** Nothing here needs concurrency while every decoder in the
    build is a dictionary lookup, and a gather over an LLM decoder would want batching and a rate
    limit rather than a bare semaphore. It waits for the decoder that needs it.
    """
    axes = [str(axis) for axis in condition.representation.axes]
    rows = []

    for row in frame.itertuples(index=False):
        utterance = Utterance(text=str(row.text), source=SOURCE)

        started = time.perf_counter_ns()
        try:
            observation = await condition.decoder.decode(utterance)
        except Exception as failure:
            observation, error = None, f"{type(failure).__name__}: {failure}"
        else:
            error = None
        elapsed_us = (time.perf_counter_ns() - started) / 1_000

        record = {"id": row.id,
                  "text": utterance.text,
                  "condition": condition.name,
                  "decoder": None if observation is None else observation.source,
                  "confidence": None if observation is None else observation.confidence,
                  "rationale": None if observation is None else observation.rationale,
                  "error": error,
                  "decode_us": elapsed_us}
        for axis in axes:
            record[f"pred_{axis}"] = (float("nan") if observation is None
                                      else float(observation.affect.values[axis]))
            # `pd.isna` rather than a bare float(): an axis the corpus does not carry is absent
            # entirely, and one it carries but never annotated is NaN or NA depending on dtype.
            label = getattr(row, axis, None)
            record[f"true_{axis}"] = float("nan") if label is None or pd.isna(label) else float(label)
        rows.append(record)

    result = pd.DataFrame(rows).set_index("id")

    # Provenance read back off the records rather than off the condition. One decoder writes one
    # source string, so more than one here means two instruments were mixed into one frame.
    sources = sorted(set(result["decoder"].dropna()))
    result.attrs = {"condition": condition.name,
                    "decoder": sources[0] if len(sources) == 1 else sources,
                    "representation": condition.representation.id,
                    "corpus": frame.attrs.get("corpus"),
                    "rows": len(result),
                    "failed": int(result["error"].notna().sum())}
    return result


In [5]:
# Decode a handful of sentences by hand - the cheapest comparison there is
#


async def probe(sentences, conditions=None, threshold: float = 0.0) -> pd.DataFrame:
    """What each condition makes of a few sentences, with the words that fired.

    For **looking at** behaviour rather than measuring it, and the fastest way to see a decoder's
    documented limits for yourself: negation is invisible, affect that is described rather than
    stated decodes to nothing, and a lexicon fires on words a hand table never had. Three sentences
    show all three.
    """
    conditions = CONDITIONS if conditions is None else conditions
    rows = []
    for condition in conditions:
        frame = pd.DataFrame({"id": [f"probe_{i:03d}" for i, _ in enumerate(sentences)],
                              "text": list(sentences)})
        result = await decode_all(frame, condition)
        axes = [str(axis) for axis in condition.representation.axes]
        for record in result.itertuples():
            fired = {axis: getattr(record, f"pred_{axis}") for axis in axes
                     if getattr(record, f"pred_{axis}") > threshold}
            rows.append({"text": record.text,
                         "condition": condition.name,
                         "fired": ", ".join(f"{a}={v:.2f}" for a, v in fired.items()) or "-",
                         "rationale": record.rationale})
    return pd.DataFrame(rows).set_index(["text", "condition"])


await probe(["I am so happy",
             "I am not happy",
             "I just got the job!",
             "I have been waiting all week to find out",
             "that is absolutely revolting"])


,,fired,rationale
text,condition,,
I am so happy,rule/handwritten,happiness=0.70,matched: happiness=happy
I am not happy,rule/handwritten,happiness=0.70,matched: happiness=happy
I just got the job!,rule/handwritten,-,no keyword matched
I have been waiting all week to find out,rule/handwritten,-,no keyword matched
that is absolutely revolting,rule/handwritten,disgust=0.80,matched: disgust=revolting
I am so happy,rule/nrc-eil,happiness=0.79,matched: happiness=happy
I am not happy,rule/nrc-eil,happiness=0.79,matched: happiness=happy
I just got the job!,rule/nrc-eil,-,no keyword matched
I have been waiting all week to find out,rule/nrc-eil,fear=0.14,matched: fear=waiting


---
## 4. Scoring

Every axis is an independent yes/no decision, so this is a **multi-label** problem and not a
classification one — a sentence may be angry and afraid at once, and both representations declared
here are non-metric, so averaging magnitudes across axes would assert a commensurability they do not
have. Only the decisions are scored.

Two exclusions do all the work, and they are different claims:

- **an axis nobody annotated** is dropped whole — BRIGHTER-eng's `disgust`, and `plutchik8/1`'s
  `anticipation` and `trust`, which no benchmark frame carries a column for;
- **a row with no ground truth at all** is dropped — a console utterance has none. A row that is
  genuinely all-rest is a *measurement of absence* and stays in, and it is the only thing that can
  produce a true negative.

The arithmetic is scikit-learn's; the decision about what is scoreable is this notebook's. That is
the right split — a hand-rolled confusion matrix in a measurement notebook is a place for a silent
mistake, and the masking is the part no library can do for us.


In [6]:
# What counts as a decision, and what is not a decision at all
#

import numpy as np
from sklearn.metrics import multilabel_confusion_matrix, precision_recall_fscore_support


def scoreable(result: pd.DataFrame, rep: AffectRepresentation) -> tuple[list[str], pd.DataFrame]:
    """The axes that have ground truth, and the rows that have any.

    An axis whose `true_` column is entirely NaN was never annotated by anyone, so it is unscoreable
    by construction and there is no second argument to fall out of step with the frame. A row with
    every `true_` NaN carries no ground truth at all and is dropped rather than counted, since
    scoring it would count every prediction as a false positive against nothing.

    **Raises when nothing is scoreable**, rather than returning an empty frame for a metric to
    average to zero. No shared axis between a decoder and a corpus is the `basic4/1`-against-an-
    `ekman6/1`-corpus case: the representations differ in what they claim is the same thing, and a
    silent 0.0 would read as a decoder that failed rather than a comparison that was never available.
    """
    axes = [axis for axis in map(str, rep.axes)
            if f"true_{axis}" in result and not result[f"true_{axis}"].isna().all()]
    if not axes:
        raise ValueError(f"no axis of {rep.id} is annotated in this benchmark - nothing to score")

    labelled = result[result[[f"true_{axis}" for axis in axes]].notna().any(axis=1)]
    if labelled.empty:
        raise ValueError("no row carries ground truth - is this a console run rather than a corpus?")
    return axes, labelled


def decisions(result: pd.DataFrame, rep: AffectRepresentation, *, threshold: float = 0.0):
    """Two boolean (row x axis) matrices: what fired, and what should have.

    `threshold` is what counts as fired, and it is the same knob as `load_tables`' `floor`. Under
    the decoder's `max` a magnitude only ever comes from the strongest matching word, so decoding at
    `floor=t` and thresholding a `floor=0` decode at `t` give identical magnitudes — sweep it here
    rather than reloading the lexicon seven times.

    **NaN compares False**, which is what makes a failed decode score as "predicted nothing" without
    a special case: its whole row is NaN, so it fires on no axis and contributes false negatives
    wherever there was truth. That is the honest reading of a decoder that could not answer.
    """
    axes, labelled = scoreable(result, rep)
    pred = labelled[[f"pred_{axis}" for axis in axes]].to_numpy() > threshold
    true = labelled[[f"true_{axis}" for axis in axes]].to_numpy() > rep.rest
    return axes, labelled, pred, true


def score(result: pd.DataFrame, rep: AffectRepresentation, *, threshold: float = 0.0) -> dict:
    """One condition at one threshold, as a single row for a comparison table.

    **Micro and macro, both reported, because they answer different questions.** Micro counts every
    (row, axis) decision once, so a common axis dominates — BRIGHTER-eng's `fear` outnumbers its
    `anger` roughly four to one. Macro gives each axis equal weight and so exposes an instrument that
    is carried by one well-covered emotion. A large gap between them is the finding, not a nuisance.

    `exact` is the share of rows whose entire axis set was right, which is the strictest reading and
    the one closest to what the agent actually consumes: the fold sees the whole vector, not one axis.

    **`silent` and `missed_entirely` are two different silences and only the second is a failure.**
    `silent` is the share of *all* scored rows where nothing fired, which includes a row the corpus
    labelled as carrying no emotion — the decoder was right to say nothing. `missed_entirely` counts
    only rows that *do* carry truth, so it is the stated-versus-inferred gap proper: the sentence
    expressed an emotion and contained no word for it. That is the gap the LLM decoder exists to
    close, so it is the number to watch across iterations, and reporting the two separately stops a
    corpus of mostly-neutral rows flattering an instrument that says nothing.

    `zero_division=0` decides what an axis that never fires and is never true is worth. Zero is the
    conservative reading and it is stated here rather than left to a default, because the alternative
    is a NaN that propagates into a mean and quietly disappears.

    The label columns come from `result.attrs`, so a row of this table says which instrument and which
    corpus produced it without the caller passing either in again.
    """
    axes, labelled, pred, true = decisions(result, rep, threshold=threshold)
    micro = precision_recall_fscore_support(true, pred, average="micro", zero_division=0)
    macro = precision_recall_fscore_support(true, pred, average="macro", zero_division=0)

    nothing_fired = (~pred).all(axis=1)
    carries_truth = true.any(axis=1)

    return {"condition": result.attrs.get("condition"),
            "corpus": result.attrs.get("corpus"),
            "decoder": result.attrs.get("decoder"),
            "threshold": threshold,
            "rows": len(result),
            "scored_rows": len(labelled),
            "scored_axes": len(axes),
            "failed": result.attrs.get("failed", 0),
            "precision": round(float(micro[0]), 3),
            "recall": round(float(micro[1]), 3),
            "f1": round(float(micro[2]), 3),
            "macro_f1": round(float(macro[2]), 3),
            "exact": round(float((pred == true).all(axis=1).mean()), 3),
            "silent": round(float(nothing_fired.mean()), 3),
            "missed_entirely": (round(float((nothing_fired & carries_truth).sum() / carries_truth.sum()), 3)
                                if carries_truth.any() else float("nan")),
            "us_median": round(float(result["decode_us"].median()), 1)}


def per_axis(result: pd.DataFrame, rep: AffectRepresentation, *, threshold: float = 0.0) -> pd.DataFrame:
    """The same decisions, one row per axis — where the micro figure came from.

    `n_true` beside `n_pred` is the first thing to read: an axis the instrument fires on ten times
    more often than the corpus labels it is over-eager rather than inaccurate, and the two failures
    want different fixes. An axis with no entries in its table can never fire, which is a fact about
    the instrument worth seeing rather than a bug — it shows up here as `n_pred` of zero.
    """
    axes, _labelled, pred, true = decisions(result, rep, threshold=threshold)
    matrices = multilabel_confusion_matrix(true, pred)

    table = pd.DataFrame({"axis": axes,
                          "n_true": true.sum(axis=0),
                          "n_pred": pred.sum(axis=0),
                          "tp": matrices[:, 1, 1],
                          "fp": matrices[:, 0, 1],
                          "fn": matrices[:, 1, 0],
                          "tn": matrices[:, 0, 0]})
    # From the counts rather than from the rounded columns above, so the F1 is the real one.
    table["precision"] = (table.tp / (table.tp + table.fp)).round(3)
    table["recall"] = (table.tp / (table.tp + table.fn)).round(3)
    table["f1"] = (2 * table.tp / (2 * table.tp + table.fp + table.fn)).round(3)
    return table.set_index("axis")


---
## 5. One condition over one benchmark

The single run, before the grid. Change the two constants and re-run.


In [7]:
# One decode, then the three views of it
#

CONDITION = CONDITIONS[1]           # rule/nrc-eil
CORPUS = "brighter-eng"

result = await decode_all(benchmarks[CORPUS], CONDITION)

display(result.attrs)
display(result[["text", "rationale", "decode_us"]].head(10))
display(pd.DataFrame([score(result, CONDITION.representation)]))
display(per_axis(result, CONDITION.representation))


{'condition': 'rule/nrc-eil',
 'decoder': 'decoder:rule:nrc-eil',
 'representation': 'ekman6/1',
 'corpus': 'brighter-eng',
 'rows': 8522,
 'failed': 0}

,text,rationale,decode_us
id,,,
eng_train_track_a_00001,"colorado, middle of nowhere.",no keyword matched,1082.833
eng_train_track_a_00002,this involved swimming a pretty large lake tha...,"matched: happiness=lake, happiness=pretty",403.334
eng_train_track_a_00003,it was one of my most shameful experiences.,matched: sadness=shameful,217.208
eng_train_track_a_00004,"after all, i had vegetables coming out my ears...",no keyword matched,226.291
eng_train_track_a_00005,then the screaming started.,"matched: anger=screaming, disgust=screaming, f...",276.459
eng_train_track_a_00006,"they don't fear death, and it seems they belie...","matched: anger=death, anger=fear, disgust=deat...",201.667
eng_train_track_a_00007,you know what happens when i get one of these ...,no keyword matched,234.167
eng_train_track_a_00008,my stomach even started giving me fits.,"matched: anger=fits, disgust=stomach, happines...",168.875
eng_train_track_a_00009,"well, as we're bowling, my dinner began to not...",matched: fear=run,164.750


,condition,corpus,decoder,threshold,rows,scored_rows,scored_axes,failed,precision,recall,f1,macro_f1,exact,silent,missed_entirely,us_median
0,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.0,8522,8522,5,0,0.457,0.42,0.437,0.408,0.142,0.323,0.298,175.4


,n_true,n_pred,tp,fp,fn,tn,precision,recall,f1
axis,,,,,,,,,
anger,1006,1860,442,1418,564,6098,0.238,0.439,0.308
fear,4822,2528,1924,604,2898,3096,0.761,0.399,0.524
happiness,2075,3247,1242,2005,833,4442,0.383,0.599,0.467
sadness,2702,2890,1438,1452,1264,4368,0.498,0.532,0.514
surprise,2498,1524,455,1069,2043,4955,0.299,0.182,0.226


In [8]:
# The threshold sweep - the magnitude floor, without reloading the lexicon
#

THRESHOLDS = (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7)

pd.DataFrame([score(result, CONDITION.representation, threshold=t) for t in THRESHOLDS])


,condition,corpus,decoder,threshold,rows,scored_rows,scored_axes,failed,precision,recall,f1,macro_f1,exact,silent,missed_entirely,us_median
0,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.0,8522,8522,5,0,0.457,0.420,0.437,0.408,0.142,0.323,0.298,175.4
1,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.2,8522,8522,5,0,0.478,0.397,0.434,0.405,0.160,0.356,0.329,175.4
2,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.3,8522,8522,5,0,0.511,0.364,0.425,0.397,0.174,0.423,0.395,175.4
3,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.4,8522,8522,5,0,0.539,0.334,0.413,0.386,0.186,0.467,0.436,175.4
4,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.5,8522,8522,5,0,0.584,0.278,0.377,0.356,0.192,0.559,0.528,175.4
5,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.6,8522,8522,5,0,0.652,0.201,0.307,0.298,0.179,0.699,0.675,175.4
6,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.7,8522,8522,5,0,0.712,0.137,0.230,0.227,0.159,0.790,0.772,175.4


---
## 6. The comparison grid

Every condition against every benchmark, at every threshold — one row per cell. This is what the
notebook exists for, and it is the thing a new decoder plugs into.

**A consistency check comes free.** `rule/nrc-eil-plutchik8` is the same lexicon and the same
algorithm as `rule/nrc-eil`, restricted to six axes instead of eight, and its two extra axes are
dropped as unscoreable — so the two rows should agree exactly on every metric. If they do not,
`restrict` is not doing what it claims and the finding is about the code rather than the decoders.


In [9]:
# Every condition against every benchmark, scored at every threshold
#


async def compare(conditions, corpora, *, thresholds=(0.0,), sample=None, random_state=0):
    """The grid, plus the decoded frames it was built from.

    **Decoded once per (condition, corpus) and scored many times.** Under the decoder's `max`,
    thresholding a `floor=0` decode at *t* gives exactly what decoding at `floor=t` would have given,
    so re-decoding per threshold would cost a pass over the corpus per threshold to produce identical
    numbers.

    `sample` takes a fixed random subset of each corpus. Every decoder in the build today is a
    dictionary lookup and needs no help, but the same grid against an LLM decoder will not be free,
    and a sweep that cannot be run small is a sweep nobody runs. `random_state` is fixed and shared,
    so every condition sees the *same* rows — a sampled comparison across different rows is not a
    comparison.

    **`attrs` is copied onto the sample explicitly.** pandas does not promise to carry it through
    `sample`, and losing it would leave every row of the grid unlabelled as to which corpus it came
    from — visible, but only if you happened to look.

    The decoded frames are returned beside the grid because error analysis needs the rows, not the
    summary, and re-decoding to get them back would be the one thing this function exists to avoid.
    """
    rows, decoded = [], {}
    for condition in conditions:
        for corpus in corpora:
            frame = benchmarks[corpus]
            if sample is not None and sample < len(frame):
                sampled = frame.sample(sample, random_state=random_state)
                sampled.attrs = dict(frame.attrs)
                frame = sampled

            result = await decode_all(frame, condition)
            decoded[(condition.name, corpus)] = result
            rows.extend(score(result, condition.representation, threshold=t) for t in thresholds)
            log.info("%s over %s: %d rows, %d failed",
                     condition.name, corpus, len(result), result.attrs["failed"])

    return pd.DataFrame(rows), decoded


In [10]:
# Run the grid
#

grid, decoded = await compare(CONDITIONS,
                              corpora=("simple-ekman6", "brighter-eng"),
                              thresholds=(0.0, 0.3, 0.5),
                              sample=None)

display(grid)
grid.pivot_table(index=["corpus", "condition"], columns="threshold", values="f1")


INFO: __main__.app.compare.line_38 - rule/handwritten over simple-ekman6: 20 rows, 0 failed
INFO: __main__.app.compare.line_38 - rule/handwritten over brighter-eng: 8522 rows, 0 failed
INFO: __main__.app.compare.line_38 - rule/nrc-eil over simple-ekman6: 20 rows, 0 failed
INFO: __main__.app.compare.line_38 - rule/nrc-eil over brighter-eng: 8522 rows, 0 failed
INFO: __main__.app.compare.line_38 - rule/nrc-eil-plutchik8 over simple-ekman6: 20 rows, 0 failed
INFO: __main__.app.compare.line_38 - rule/nrc-eil-plutchik8 over brighter-eng: 8522 rows, 0 failed


,condition,corpus,decoder,threshold,rows,scored_rows,scored_axes,failed,precision,recall,f1,macro_f1,exact,silent,missed_entirely,us_median
0,rule/handwritten,simple-ekman6,decoder:rule:handwritten,0.0,20,20,6,0,0.867,0.765,0.812,0.836,0.750,0.300,0.176,9.7
1,rule/handwritten,simple-ekman6,decoder:rule:handwritten,0.3,20,20,6,0,0.867,0.765,0.812,0.836,0.750,0.300,0.176,9.7
2,rule/handwritten,simple-ekman6,decoder:rule:handwritten,0.5,20,20,6,0,0.867,0.765,0.812,0.836,0.750,0.300,0.176,9.7
3,rule/handwritten,brighter-eng,decoder:rule:handwritten,0.0,8522,8522,5,0,0.771,0.028,0.054,0.057,0.120,0.945,0.941,6.3
4,rule/handwritten,brighter-eng,decoder:rule:handwritten,0.3,8522,8522,5,0,0.771,0.028,0.054,0.057,0.120,0.945,0.941,6.3
5,rule/handwritten,brighter-eng,decoder:rule:handwritten,0.5,8522,8522,5,0,0.842,0.022,0.042,0.044,0.114,0.961,0.959,6.3
6,rule/nrc-eil,simple-ekman6,decoder:rule:nrc-eil,0.0,20,20,6,0,0.500,0.706,0.585,0.565,0.350,0.300,0.235,179.7
7,rule/nrc-eil,simple-ekman6,decoder:rule:nrc-eil,0.3,20,20,6,0,0.522,0.706,0.600,0.579,0.350,0.300,0.235,179.7
8,rule/nrc-eil,simple-ekman6,decoder:rule:nrc-eil,0.5,20,20,6,0,0.688,0.647,0.667,0.653,0.600,0.400,0.294,179.7
9,rule/nrc-eil,brighter-eng,decoder:rule:nrc-eil,0.0,8522,8522,5,0,0.457,0.420,0.437,0.408,0.142,0.323,0.298,173.2


threshold                               0.0    0.3    0.5
corpus        condition                                  
brighter-eng  rule/handwritten        0.054  0.054  0.042
              rule/nrc-eil            0.437  0.425  0.377
              rule/nrc-eil-plutchik8  0.437  0.425  0.377
simple-ekman6 rule/handwritten        0.812  0.812  0.812
              rule/nrc-eil            0.585  0.600  0.667
              rule/nrc-eil-plutchik8  0.585  0.600  0.667

---
## 7. Diagnostic — the signal `ekman6/1` has no axis for

`plutchik8/1` is **not a third condition in an accuracy comparison**: two of its axes have no ground
truth in any benchmark frame and never will, so anyone reporting accuracy over eight axes has scored
two of them against nothing. What it *can* answer is how much lexical signal the narrower
representation throws away — how often the strongest word in a sentence lands on `anticipation` or
`trust`, and how often those are the only axes that fire at all.


In [11]:
# How often the strongest signal lands where the narrower representation cannot follow
#


def off_representation(result: pd.DataFrame, rep: AffectRepresentation, *,
                       beyond, threshold: float = 0.0) -> dict:
    """The share of rows whose signal falls outside `beyond`'s complement.

    `beyond` names the axes the *narrower* representation does not have. Two different questions,
    both worth having: `top_beyond` is how often the strongest reading is one `ekman6/1` cannot
    express, and `only_beyond` is how often dropping those axes silences the row completely — the
    stronger claim, since a row that also fired on `fear` still decodes to something.

    Failed decodes are floored below any threshold before `argmax`, so a NaN row cannot win a
    comparison it never entered.
    """
    axes = np.array([str(axis) for axis in rep.axes])
    values = np.nan_to_num(result[[f"pred_{axis}" for axis in axes]].to_numpy(), nan=-1.0)
    fired = values > threshold
    outside = np.isin(axes, list(beyond))

    top = np.where(fired.any(axis=1), axes[values.argmax(axis=1)], "")
    return {"rows": len(result),
            "any_fired": round(float(fired.any(axis=1).mean()), 3),
            "top_beyond": round(float(np.isin(top, list(beyond)).mean()), 3),
            "only_beyond": round(float((fired[:, outside].any(axis=1)
                                        & ~fired[:, ~outside].any(axis=1)).mean()), 3)}


BEYOND_EKMAN6 = ("anticipation", "trust")

pd.DataFrame([off_representation(decoded[("rule/nrc-eil-plutchik8", corpus)], PLUTCHIK8,
                                 beyond=BEYOND_EKMAN6) | {"corpus": corpus}
              for corpus in ("simple-ekman6", "brighter-eng")]).set_index("corpus")


,rows,any_fired,top_beyond,only_beyond
corpus,,,,
simple-ekman6,20,0.750,0.050,0.050
brighter-eng,8522,0.768,0.228,0.072


---
## 8. Error analysis

The score says how much is wrong; this says **what** is wrong and why. `rationale` is the field that
makes it possible — it carries the words that fired, including one that lost the `max`, so a
false positive can be read back to the word that caused it. "Good grief" scoring as mild happiness
looks identical to a sentence that really was mildly happy until you can see that the word was
"good".


In [12]:
# One row per wrong axis decision, with the words that caused it
#


def mistakes(result: pd.DataFrame, rep: AffectRepresentation, *, threshold: float = 0.0) -> pd.DataFrame:
    """Long form: every (row, axis) pair the decoder got wrong.

    Long rather than a filtered copy of the wide frame, because a mistake **is** a (row, axis) pair —
    a sentence can be a false positive on one axis and a false negative on another in the same
    breath, and a wide row cannot say which of its columns is being complained about.
    """
    axes, labelled, _pred, _true = decisions(result, rep, threshold=threshold)

    parts = []
    for axis in axes:
        fired = labelled[f"pred_{axis}"] > threshold
        truth = labelled[f"true_{axis}"] > rep.rest
        for kind, mask in (("false positive", fired & ~truth), ("false negative", ~fired & truth)):
            part = labelled[mask]
            parts.append(pd.DataFrame({"kind": kind,
                                       "axis": axis,
                                       "text": part["text"],
                                       "magnitude": part[f"pred_{axis}"].round(2),
                                       "rationale": part["rationale"]}))

    return pd.concat(parts).sort_values(["kind", "axis"])


wrong = mistakes(result, CONDITION.representation)
false_positives = wrong[wrong.kind == "false positive"]

display(wrong.groupby(["axis", "kind"]).size().unstack(fill_value=0))
display(false_positives.sample(min(10, len(false_positives)), random_state=0))


kind,false negative,false positive
axis,,
anger,564,1418
fear,2898,604
happiness,833,2005
sadness,1264,1452
surprise,2043,1069


,kind,axis,text,magnitude,rationale
id,,,,,
eng_test_track1_2518,false positive,anger,' but i don't know the worst is about to come....,0.67,"matched: anger=hit, sadness=gone"
eng_test_track_c_01593,false positive,sadness,hed sat down for a moment to rest his weary le...,0.50,"matched: happiness=rest, sadness=down, sadness..."
eng_train_track_a_01975,false positive,fear,i was doing well emotionally last night and wa...,0.33,"matched: anger=feeling, disgust=feeling, fear=..."
eng_test_track1_1879,false positive,sadness,but my ankle has just about completely stopped...,0.62,"matched: anger=hurting, fear=hurting, sadness=..."
eng_test_track1_284,false positive,fear,they dance and shiver and grin and laugh.,0.48,"matched: anger=shiver, fear=shiver, happiness=..."
eng_test_track_c_00716,false positive,happiness,it was super awkward.,0.61,matched: happiness=super
eng_train_track_a_00229,false positive,anger,apparently the aftermath of the clogged toilet...,0.36,"matched: anger=aftermath, disgust=aftermath, d..."
eng_test_track_c_01329,false positive,sadness,"this is what happened: it was bright, at firs...",0.55,"matched: anger=storm, sadness=dark"
eng_test_track_c_01980,false positive,sadness,it sounded very deep like almost demonic.,0.66,"matched: anger=demonic, disgust=demonic, fear=..."


In [13]:
# Rows that expressed an emotion and fired nothing - the stated-versus-inferred gap, in sentences
#
# The keyword decoder decodes affect a person has STATED and cannot infer it from a situation. These
# are the rows where that shows, and the filter is the point: a row the corpus labelled as carrying
# no emotion also fires nothing, and the decoder was RIGHT about it. Only rows with truth belong
# here. They are the case the LLM decoder exists for, so they are worth reading rather than counting.

axes_scored, labelled_rows = scoreable(result, CONDITION.representation)
true_columns = [f"true_{a}" for a in axes_scored]

fired_nothing = ~(labelled_rows[[f"pred_{a}" for a in axes_scored]] > 0.0).any(axis=1)
carries_truth = (labelled_rows[true_columns] > CONDITION.representation.rest).any(axis=1)
gap = labelled_rows[fired_nothing & carries_truth]

print(f"{len(gap):,} of {int(carries_truth.sum()):,} rows carrying an emotion decoded to nothing "
      f"({len(gap) / int(carries_truth.sum()):.1%}) - "
      f"{int((fired_nothing & ~carries_truth).sum()):,} further rows were correctly silent")

display(gap[["text", *true_columns]].sample(min(10, len(gap)), random_state=0))


2,291 of 7,676 rows carrying an emotion decoded to nothing (29.8%) - 460 further rows were correctly silent


,text,true_anger,true_fear,true_happiness,true_sadness,true_surprise
id,,,,,,
eng_test_track1_442,never saw that floor again either.,0.0,1.0,0.0,1.0,1.0
eng_train_track_a_01636,yeah it was a little embarrassing.,0.0,1.0,0.0,1.0,0.0
eng_train_track_a_01694,anyway it was weird.,0.0,1.0,0.0,0.0,1.0
eng_test_track_c_02443,"i was poised, well balanced and ready for him.",0.0,1.0,1.0,0.0,0.0
eng_train_track_a_00204,a rather regular occurrence happened at night ...,0.0,1.0,0.0,0.0,0.0
eng_test_track_c_02686,my eyes are staring up at the ceiling but i ca...,0.0,1.0,0.0,0.0,0.0
eng_test_track1_030,"now, teresa claims that the rapport i have wit...",1.0,1.0,0.0,1.0,0.0
eng_test_track1_2177,but he is my... just hold my hand through this.,0.0,1.0,0.0,1.0,0.0
eng_train_track_a_01987,3rd i really don't know... any of the main cha...,0.0,1.0,0.0,0.0,0.0


---
## 9. Persisting a result

`data_results/` is gitignored, so this is a working artefact rather than a record — but a comparison
worth keeping is worth being able to identify later. The provenance travels inside the parquet as
`attrs`, the same trick the benchmark frames use, so there is no sidecar file to lose: the decoder
source strings (not the notebook's labels), the corpora with their own `attrs`, the thresholds and
the sample size.

Filenames are timestamped, so nothing overwrites anything.


In [14]:
# Write the grid with enough provenance to know what it measured
#

from datetime import UTC, datetime
from pathlib import Path


def save(grid_df: pd.DataFrame, decoded_frames: dict, *, name: str = "decoder_comparison") -> Path:
    """The grid to parquet, with provenance in `attrs` rather than in the filename.

    The decoder identity written here is the `source` string off the records — `decoder:rule:` plus
    the lexicon that was actually read — never the notebook's own label, so a result stays traceable
    after the labels have been rewritten. Same reasoning as `Lexicon` returning its source beside its
    tables: the name and the thing it names travel together, or they eventually disagree.
    """
    stamp = datetime.now(UTC).strftime("%Y-%m-%dT%H%M%SZ")
    path = DATA_RESULTS / f"{name}.{stamp}.parquet"

    saved = grid_df.copy()
    saved.attrs = {"written_at": stamp,
                   "lexicon": lexicon.source,
                   "decoders": sorted({str(frame.attrs["decoder"]) for frame in decoded_frames.values()}),
                   "corpora": {corpus: dict(frame.attrs) for corpus, frame in benchmarks.items()},
                   "rows_decoded": {f"{cond}|{corpus}": int(frame.attrs["rows"])
                                    for (cond, corpus), frame in decoded_frames.items()}}
    saved.to_parquet(path)
    log.info("wrote %s", path)
    return path


saved_to = save(grid, decoded)
display(pd.read_parquet(saved_to).attrs)


INFO: __main__.app.save.line_27 - wrote /Users/stuartgow/PhD Project/Repo asa_research_prototype/data_results/decoder_comparison.2026-08-14T111434Z.parquet


{'written_at': '2026-08-14T111434Z',
 'lexicon': 'nrc-eil',
 'decoders': ['decoder:rule:handwritten', 'decoder:rule:nrc-eil'],
 'corpora': {'simple-ekman6': {'corpus': 'simple-ekman6',
   'representation': 'ekman6/1',
   'axis_map': {},
   'unannotated_axes': [],
   'rows': 20},
  'brighter-eng': {'corpus': 'brighter-eng',
   'representation': 'ekman6/1',
   'axis_map': {'joy': 'happiness'},
   'unannotated_axes': ['disgust'],
   'rows': 8522}},
 'rows_decoded': {'rule/handwritten|simple-ekman6': 20,
  'rule/handwritten|brighter-eng': 8522,
  'rule/nrc-eil|simple-ekman6': 20,
  'rule/nrc-eil|brighter-eng': 8522,
  'rule/nrc-eil-plutchik8|simple-ekman6': 20,
  'rule/nrc-eil-plutchik8|brighter-eng': 8522}}